# 05 — Iterative DiffusionDPO

Why iterative DPO? Standard DPO is **offline**: preferences are collected once
from the initial LoRA model. As the policy updates, the preference dataset
becomes stale — the policy shifts but the data doesn't.

Iterative DPO alternates between:
1. **Rollout**: generate fresh video pairs with the current policy
2. **Score**: reward model labels them (chosen, rejected)
3. **Update**: DPO gradient steps on fresh on-policy pairs
4. **Evaluate**: win rate on held-out prompts vs baseline

This is the exact same loop as `14_iterative_dpo.ipynb` in the text RLHF repo,
with `rollout()` generating video pairs instead of text responses.

**Expected win rate trajectory:** 50% → 55% → 62% → 67% over 3-5 iterations.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Visualize the iterative DPO loop structure
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

loop_text = """
┌─────────────────────────────────────────────────────────┐
│             Iterative DiffusionDPO Loop                │
│                                                         │
│  π₀ (LoRA fine-tuned)                                  │
│      ↓                                                  │
│  Iteration 1:                                           │
│    [Rollout] Generate N pairs from π₀                  │
│    [Score]   Reward model → (chosen, rejected)         │
│    [DPO]     K gradient steps → π₁                    │
│    [Eval]    Win rate π₁ vs π₀ on held-out prompts     │
│      ↓                                                  │
│  Iteration 2:                                           │
│    [Rollout] Generate N pairs from π₁                  │
│    [Score]   Reward model → (chosen, rejected)         │
│    [DPO]     K gradient steps → π₂                    │
│    [Eval]    Win rate π₂ vs π₀                         │
│      ↓                                                  │
│  ...repeat until win_rate ≥ target or max_iterations   │
└─────────────────────────────────────────────────────────┘
"""
ax.text(0.05, 0.95, loop_text, transform=ax.transAxes,
        fontfamily='monospace', fontsize=9, va='top')
plt.tight_layout()
plt.show()

In [ ]:
# Buffer strategy comparison
# The key trade-off: recency vs. data quantity

iterations = np.arange(1, 6)
# Simulated win rates for different buffer strategies
win_rate_current = [0.50, 0.54, 0.57, 0.59, 0.60]   # noisy but fresh
win_rate_rolling2 = [0.52, 0.58, 0.63, 0.66, 0.68]  # balanced
win_rate_full = [0.51, 0.56, 0.62, 0.65, 0.66]       # more data but distribution mismatch
win_rate_offline = [0.53, 0.53, 0.54, 0.54, 0.54]    # no iterative DPO (flat)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(iterations, win_rate_current, 'o--', label='current (this iter only)', color='#C44E52')
ax.plot(iterations, win_rate_rolling2, 's-', label='rolling2 (last 2 iters)', color='#4C72B0', linewidth=2)
ax.plot(iterations, win_rate_full, '^--', label='full (all iters)', color='#55A868')
ax.plot(iterations, win_rate_offline, 'x:', label='offline DPO (no iteration)', color='gray')
ax.axhline(0.5, color='black', linestyle=':', alpha=0.5, label='Random baseline')
ax.axhline(0.70, color='red', linestyle='--', alpha=0.5, label='Target (0.70)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Win rate vs. baseline')
ax.set_title('Iterative DPO: buffer strategy comparison (simulated)')
ax.set_ylim(0.40, 0.80)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('rolling2 is the best trade-off: recent data for alignment + enough diversity.')
print('Same finding as the text RLHF iterative DPO ablation (Notebook 14, prior repo).')

In [ ]:
# Load and display real iteration history if available
import os

history_path = '../checkpoints/iterative_dpo/history.json'
if os.path.exists(history_path):
    with open(history_path) as f:
        history = json.load(f)
    df = pd.DataFrame(history)
    print('Real iterative DPO history:')
    print(df.to_string(index=False))
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['iteration'], df['win_rate'], 'o-', color='#4C72B0', linewidth=2, markersize=8)
    ax.axhline(0.5, color='gray', linestyle='--', label='Random')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Win rate vs. baseline')
    ax.set_title('Iterative DPO win rate progression')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f'History file not found at {history_path}.')
    print('Run: python scripts/run_iterative_dpo.py --config configs/iterative_dpo_config.yaml')